In [ ]:
##! pip install pandas-ta==0.2.23b
import math
import pdb
from fyers_apiv3 import fyersModel
import pyotp
import pandas as pd
import requests as req
import datetime as dt
from urllib.parse import parse_qs, urlparse
import time
import talib as ta
import warnings
import joblib
from time import sleep
warnings.filterwarnings("ignore")
import numpy as np
import glob

In [ ]:
data_source = r"F:\Cources\Office\VSINTERN\Histoty\5"
path = glob.glob(f"{data_source}\*.csv")[:2]
path

In [ ]:
data = [pd.read_csv(i) for i in path] if type(path) == list else pd.read_csv(path)
data.head()

In [ ]:
def indicator(data):
    """
    Code your indicator logic here
    
    
    Should:
        return list of dataframe if data passed is list of dataframe. assuming that data is of different instrument. 
    """
    
    return data
indicator_data  = [indicator(i) for i in data] if type(data) == list else indicator(data)

In [ ]:
def signal_generator(data):
    """
    Code your strategy scaner aand signal generator logic here
    
    
    Should:
        return list of dataframe if data passed is list of dataframe. assuming that data is of different instrument. 
    """
    return data

generated_signal  = [signal_generator(i) for i in indicator_data] if type(indicator_data) == list else signal_generator(indicator_data)

In [ ]:
def consolidate_data(data_list):
    """
    consolidate each data in data_list
    
    """
    if type(data_list) == list:
        data = pd.concat(data_list)
        data['Date'] = pd.to_datetime(data['Date'])
        data.sort_values(by='Date', inplace=True)
        return data
    else:
        return data_list

consolidated_data = consolidate_data(indicator_data)


In [ ]:
class Backtesting_Engine:
    def __init__(self, data=None, initial_capital=1000000, lot_size=1, sl=0.02, tgt=0.04):
        self.data = data
        self.capital = initial_capital
        self.lot_size = lot_size
        self.position = {}
        columns=['Date','Symbol',"Price","Qty","SL","TGT","Type","Transaction Cost","Amount","Y"]
        self.ledger = pd.DataFrame(columns=columns)
        self.__brokerages__()
        

    def __brokerages__(self):
        """
        Trading charges are taken from Zerodha's brokerage Charges portal
        url: https://zerodha.com/charges/#tab-equities
        """
        self.SEBI = 0.0001/100 #on both side
        self.Transaction_cost = 0.00297/100 #on both side
        self.brokerage = 0.03/100 #On both side
        self.GST = 18/100 #on brokerage, SEBI, Transaction cost
        self.STT = 0.025/100 #on both side
        
        
        self.stamp = 0.003/100 #on buy side


    def transaction_cost_caluclation(self, price, type_of_trade, quantity):
        """ArithmeticError
        Return : array as >> [SEBI,TC, Brokerage, GST, STT, Stamp Duty, Total Cost]
        """
        turnover = price * quantity
        #print(f"Turnover: {turnover}")
        #Cost FACtor
        cost_factor = np.array([self.SEBI, self.Transaction_cost])
        #Cost of SEBU, Transaction cost and Brokerage
        cost = turnover * cost_factor
        #Brokrage cost with a cap of 30
        cost = np.append(cost, np.min([self.brokerage * turnover,30]))
        #GST on brokerage|SEBI|Transaction cost
        cost = np.append(cost, self.GST * cost.sum())
        #STT cost
        cost = np.append(cost, self.STT * turnover)
        #Stamp duty
        if type_of_trade == 'buy':
            cost = np.append(cost, self.stamp * turnover)
        else:
            cost = np.append(cost, 0)
        cost = np.append(cost, cost.sum())
        return cost
    
    def simulator(self):
        for index, row in self.data.iterrows():
            symb = row['Symbol']
            entry_price = row['Close']
            signal = row['signal']
            _date_ = row['Date']
            rtp = 0.01
            if symb not in self.position:
                loss_per_trade = self.capital*rtp
                if signal == 1:
                    sl = entry_price * (1-0.02)
                    tgt = entry_price * (1+0.04)
                    
                    qty = math.floor(loss_per_trade/abs(entry_price-sl))
                    if qty>0:
                        cost = self.transaction_cost_caluclation(entry_price, 'buy', qty)
                        amuont = cost[-1] + entry_price*qty
                        self.position[symb] = [_date_,symb, entry_price,qty, sl, tgt, 'buy', cost,-1*amuont,"Entry"]
                        self.ledger.loc[len(self.ledger)] = self.position[symb]
                        self.capital -= (cost[-1]+entry_price*qty)
                elif signal == -1:
                    sl = entry_price * (1+0.02)
                    tgt = entry_price * (1-0.04)
                    
                    qty = math.floor(loss_per_trade/abs(entry_price-sl))
                    if qty>0:
                        cost = self.transaction_cost_caluclation(entry_price, 'sell', qty)
                        amuont = cost[-1] + entry_price*qty
                        self.position[symb] = [_date_,symb, entry_price,qty, sl, tgt, 'sell', cost,amuont,"Entry"]
                        self.ledger.loc[len(self.ledger)] = self.position[symb]
                        self.capital -= cost[-1]
                        self.capital += entry_price*qty
            elif symb in self.position:
                position_data = self.position[symb]
                type_ = position_data[6]
                qty = position_data[3]
                if type_ == 'buy':
                    if position_data[4] >= entry_price:
                        #SL Hit
                        #['Date','Symbol',"Price","Qty","SL","TGT","Type","Transaction Cost","Amount"]
                        cost = self.transaction_cost_caluclation(entry_price, 'sell', qty)
                        self.capital-= cost[-1]
                        self.capital += entry_price*qty
                        amuont = entry_price*qty - cost[-1]
                        self.ledger.loc[len(self.ledger)] = [_date_,symb,entry_price,qty,0,0,"sell",cost,amuont,"SL"]
                        del self.position[symb]
                    elif position_data[5] <= entry_price:
                        cost = self.transaction_cost_caluclation(entry_price, 'sell', qty)
                        self.capital-= cost[-1]
                        self.capital += entry_price*qty
                        amuont = entry_price*qty - cost[-1]
                        self.ledger.loc[len(self.ledger)] = [_date_,symb,entry_price,qty,0,0,"sell",cost,amuont,"TGT"]
                        del self.position[symb]
                elif type_ == 'sell':
                    if position_data[4] <= entry_price:
                        #SL Hit
                        #['Date','Symbol',"Price","Qty","SL","TGT","Type","Transaction Cost","Amount"]
                        cost = BE.transaction_cost_caluclation(entry_price, 'buy', qty)
                        self.capital-= cost[-1]
                        self.capital -= entry_price*qty
                        amuont = entry_price*qty - cost[-1]
                        self.ledger.loc[len(self.ledger)] = [_date_,symb,entry_price,qty,0,0,"buy",cost,-amuont,"SL"]
                        del self.position[symb]
                    elif position_data[5] >= entry_price:
                        cost = self.transaction_cost_caluclation(entry_price, 'buy', qty)
                        self.capital-= cost[-1]
                        self.capital -= entry_price*qty
                        amuont = entry_price*qty - cost[-1]
                        self.ledger.loc[len(self.ledger)] = [_date_,symb,entry_price,qty,0,0,"buy",cost,-amuont,"TGT"]
                        del self.position[symb]

        return self.ledger

BE = Backtesting_Engine(data=consolidated_data)
ledger  = BE.simulator()
#list(BE.transaction_cost_caluclation(1000, 'buy', 100000))